# Update-method comparison — Colab runner

Runs the fixed comparison end to end. **Runtime → Change runtime type → A100** (L4 works, **T4 does not**: Gemma-2-9B bf16 needs >16 GB and the adapter forbids offload).

Order matters. Each section refuses to run if the previous receipt is missing. Re-running is safe — finished shards are skipped.


## 0 · Paths and upload

Upload `UPDATE_METHOD_INPUTS.zip` and `UPDATE_METHOD_RESULTS.zip`, or mount Drive.

In [ ]:
import os, subprocess, sys, json, pathlib

BASE = "/content/work"
PKG  = f"{BASE}/UPDATE_METHOD_INPUTS"
OUT  = f"{BASE}/UPDATE_METHOD_RESULTS"
HF   = f"{BASE}/hf"
REPO = f"{BASE}/programmable-kv"
os.makedirs(BASE, exist_ok=True); os.makedirs(HF, exist_ok=True)

# from google.colab import files; files.upload()
!cd {BASE} && unzip -o -q UPDATE_METHOD_INPUTS.zip && unzip -o -q UPDATE_METHOD_RESULTS.zip

# code/ must sit beside inputs.py / metrics.py / canonical/
!cp {OUT}/code/*.py {PKG}/
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Model access

`google/gemma-2-9b-it` is gated. Accept the licence on your own HF account, then add a read token to **Colab Secrets** as `HF_TOKEN` (key icon, left sidebar) and enable notebook access.

Do not paste a token into a cell or a file — `build_manifest.py` scans for them and will refuse to finalize the package.

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token loaded:", bool(os.environ.get("HF_TOKEN")))

## 2 · Environment

Recorded scientific runtime: Python 3.12.3 / torch 2.11.0+cu128. Colab will likely differ — **record what you actually get**, don't hide the deviation.

In [ ]:
!pip install -q -r {PKG}/requirements-inference.txt
import torch, transformers
env = dict(python=sys.version.split()[0], torch=torch.__version__,
           cuda=torch.version.cuda, transformers=transformers.__version__,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print(json.dumps(env, indent=2))
pathlib.Path(f"{OUT}/raw").mkdir(parents=True, exist_ok=True)
pathlib.Path(f"{OUT}/raw/COLAB_ENVIRONMENT.json").write_text(json.dumps(env, indent=2))

## 3 · Pinned snapshots

`Backbone` loads with `local_files_only=True`, so these must land first. ~35 GB total.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3-8B",        revision="b968826d9c46dd6066d109eabc6255188de91218", cache_dir=HF)
snapshot_download("google/gemma-2-9b-it", revision="11c9b309abf73637e4b6f9a3fa1e92e615547819", cache_dir=HF)
!du -sh {HF}

## 4 · Package verification + CPU checks (Step 5)

Expected template hash: `2d88092b1c5dfa5da26f59ed7262713fb1b28d4f920579aa991ca5d5d0467c68`. A mismatch means the erratum template was edited — stop and re-freeze deliberately.

In [ ]:
%cd {PKG}
!python verify_package.py && echo "--- verify OK ---"
!python check_cpu.py --output {OUT}/raw/CPU_CHECK_colab.json
!python preflight.py --stage cpu --package {PKG} --output {OUT}/raw/PREFLIGHT_CPU.json
print(json.load(open(f"{OUT}/raw/PREFLIGHT_CPU.json"))["erratum_template"]["sha256"])

## 5 · Stage 1 external anchor (Step 4)

If this fails for a concrete compatibility reason: record the exact error and **stop this branch**. Do not substitute an invented method.

In [ ]:
!git clone -q https://github.com/19PINE-AI/programmable-kv {REPO}
!cd {REPO} && git checkout -q e9085eafcc6c83e60c548de69060c4bd5b210c96 && git rev-parse HEAD
!pip install -q -e {REPO}

%cd {PKG}
!python stage1_worked_example.py --repo {REPO} --model Qwen/Qwen3-8B \
   --output {OUT}/external_reproduction/outputs/worked_example.json \
   > {OUT}/external_reproduction/stdout.txt 2> {OUT}/external_reproduction/stderr.txt
!tail -30 {OUT}/external_reproduction/stdout.txt; echo "--- stderr ---"; tail -20 {OUT}/external_reproduction/stderr.txt

## 6 · Model preflight — THE GATE (Step 7)

Read `refresh_census` before going further.

`FIELD_PLUS_LATEST_ERRATUM` is the primary baseline. Its field-refresh half only executes when swapping `in favor of` ↔ `opposed to` is **token-length-preserving**. If `fallback_rate` is high, that arm collapses toward `LATEST_ERRATUM` and the headline must say so.

This does not change the plan — all arms still run. It changes how the primary contrast is reported.

In [ ]:
%cd {PKG}
!python preflight.py --stage model --package {PKG} --cache {HF} --actor qwen  --output {OUT}/raw/PREFLIGHT_MODEL_qwen.json
!python preflight.py --stage model --package {PKG} --cache {HF} --actor gemma --output {OUT}/raw/PREFLIGHT_MODEL_gemma.json

for a in ("qwen","gemma"):
    c = json.load(open(f"{OUT}/raw/PREFLIGHT_MODEL_{a}.json"))["refresh_census"]
    print(f"\n=== {a} ===")
    print("feasible", c["feasible"], "/", c["scenes"], " fallback_rate", c["fallback_rate"])
    print("reasons:", c["reasons"])
    print(c["interpretation"])

## 7 · DEV smoke + freeze (Step 7)

DEV is for adapter checks only, **never** efficacy selection. Do not read it as a result.

In [ ]:
%cd {PKG}
!python run_comparison.py --package {PKG} --raw {OUT}/raw/dev --cache {HF} --panel DEV --limit 2
!python build_manifest.py --root {OUT} --output {OUT}/MANIFEST.json --stage frozen

## 8 · Full fixed comparison (Step 8)

19,392 context builds, ~531k scored answers. Sharded by `(actor, panel, method, seed)`; finished shards are skipped.

**Re-run this cell after every disconnect** until no shards remain. `--max-seconds` stops cleanly before the session limit.

`ORIGIN` carries the primary endpoint — run it first if compute is tight.

In [ ]:
%cd {PKG}
!python run_comparison.py --package {PKG} --raw {OUT}/raw --cache {HF} --panel ORIGIN --max-seconds 32000

In [ ]:
# then the rest
%cd {PKG}
!python run_comparison.py --package {PKG} --raw {OUT}/raw --cache {HF} --max-seconds 32000

In [ ]:
# what is still missing — this IS the record of identified missing cells
!python run_comparison.py --package {PKG} --raw {OUT}/raw --report-gaps

## 9 · Tables, figure, manifest (Step 9)

In [ ]:
%cd {PKG}
!python analysis.py --package {PKG} --raw {OUT}/raw --tables {OUT}/tables
!python figure.py --tables {OUT}/tables --output {OUT}/figure/comparison.png
!python build_manifest.py --root {OUT} --output {OUT}/MANIFEST.json --stage final

from IPython.display import Image, display
display(Image(f"{OUT}/figure/comparison.png"))
print(json.dumps(json.load(open(f"{OUT}/tables/primary_contrasts.json"))["status"], indent=2))

## 10 · Package for return

`build_manifest.py` fails closed on token-like strings and machine paths. Fix any finding before downloading.

In [ ]:
!cd {BASE} && rm -f UPDATE_METHOD_RESULTS_filled.zip && zip -qr UPDATE_METHOD_RESULTS_filled.zip UPDATE_METHOD_RESULTS \
    -x "*/__pycache__/*" "*/.ipynb_checkpoints/*"
!ls -la {BASE}/UPDATE_METHOD_RESULTS_filled.zip
from google.colab import files; files.download(f"{BASE}/UPDATE_METHOD_RESULTS_filled.zip")